In [ ]:
%load_ext autoreload
%autoreload 2

In [17]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [18]:
from src.trainer import SmoothTrainer, IntervalTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead, set_seed
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_IT_CONFIG as CONFIG

### Task Incremental Learning

In [19]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [20]:
interval_trainer = IntervalTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="TIL",
    seed=SEED,
)

interval_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=CONFIG["batch_size"],
    epochs=CONFIG["epochs"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    context_id=0,
)
interval_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

Training Epochs: 100%|█| 5/5 [00:06<00:00,  1.31s/it, val_loss=0.0145, val_acc=0


Test Results: [(0.0342, 0.9865), (1.3716, 0.6244), (2.7784, 0.4564), (3.0867, 0.0595), (2.8978, 0.4493)] (Avg: (2.0337, 0.5152))
Initial acc constraint violation: -0.1262 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.86
Initial bbox:  Obj=14.94,  Size=92320.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|█| 200/200 [00:09<00:00, 20.34it/s, size=92439.24, obj=14.960, min_soft_acc


Final bbox:  Obj=14.96,  Size=92439.24,  Min acc hard=0.53,  Min acc soft=0.53
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['92324.77', '92344.61', '92357.80', '92370.55', '92382.98', '92390.55', '92402.26', '92414.54', '92426.86', '92439.24']
Checkpoint certificates: ['0.73', '0.52', '0.51', '0.49', '0.50', '0.53', '0.53', '0.53', '0.53', '0.53']
----------------------- Finished Computing Rashomon set ------------------------


In [21]:
smooth_trainer = SmoothTrainer(
    interval_trainer.model,
    400,
    0.9,
    paradigm="TIL",
    seed=SEED
)

smooth_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=64,
    epochs=5,
    lr=0.01,
    context_id=0
)
smooth_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

smooth_trainer.compute_lid(test_tasks[0], context_id=0, bounded_model=interval_trainer.bounds[0])

smooth_trainer.train(
    train_tasks[1],
    val_tasks[1],
    batch_size=64,
    epochs=5,
    lr=0.01,
    context_id=1
)
smooth_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))


Training Epochs:   0%| | 0/5 [00:00<?, ?it/s, train_loss=0.3999, val_loss=0, val

Training Epochs: 100%|█| 5/5 [00:06<00:00,  1.31s/it, train_loss=0.0007, val_los


Test Results: [(0.0281, 0.9905), (2.0516, 0.4304), (2.4819, 0.3115), (2.4974, 0.3283), (2.3077, 0.4984)] (Avg: (1.8733, 0.5118))


100%|████████████████████████████████████████| 100/100 [00:00<00:00, 656.82it/s]


Estimated success:  1.0  Lower bound:  0.9637833073548235
Certified L2 radius: 1.7964


...

100%|████████████████████████████████████████| 100/100 [00:00<00:00, 652.40it/s]


Estimated success:  0.95  Lower bound:  0.8871650888945373
Certified L2 radius: 42.4056
Max rad:  44.909612808251616  inflate:  25
Last rad:  42.40560815947524  last inflate:  35


Training Epochs: 100%|█| 5/5 [00:10<00:00,  2.10s/it, train_loss=0.0076, val_los


Test Results: [(0.0261, 0.992), (0.0214, 0.994), (2.3069, 0.3498), (2.4957, 0.3181), (2.3157, 0.5422)] (Avg: (1.4332, 0.6392))


[(0.0261, 0.992),
 (0.0214, 0.994),
 (2.3069, 0.3498),
 (2.4957, 0.3181),
 (2.3157, 0.5422)]

### Domain Incremental Learning

In [4]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=2, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [5]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [6]:
interval_trainer = IntervalTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
    seed=SEED,
)

interval_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=CONFIG["batch_size"],
    epochs=CONFIG["epochs"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)
interval_trainer.test(test_tasks)

interval_trainer.compute_rashomon_set(test_tasks[0])

Training Epochs: 100%|█| 5/5 [00:07<00:00,  1.40s/it, val_loss=0.0110, val_acc=0


Test Results: [(0.0235, 0.9915), (0.5784, 0.8381), (1.2203, 0.5708), (3.1147, 0.4365), (5.4571, 0.1852)] (Avg: (2.0788, 0.6044))
Initial acc constraint violation: -0.1250 (Positive = violated)
[tensor(0.8675, device='cuda:0'), tensor(0.8675, device='cuda:0'), tensor(0.8675, device='cuda:0'), tensor(0.8675, device='cuda:0'), tensor(0.8675, device='cuda:0')]
Number of model parameters: 1563
Computing Rashomon set with min acc limit: 0.87
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|█| 200/200 [00:10<00:00, 19.46it/s, size=60.69, obj=0.039, min_soft_acc=1.0


Final bbox:  Obj=0.04,  Size=60.69,  Min acc hard=0.85,  Min acc soft=0.85
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['15.76', '27.43', '36.92', '41.15', '44.83', '43.92', '51.34', '56.86', '56.60', '60.69']
Checkpoint certificates: ['0.87', '0.89', '0.86', '0.86', '0.86', '0.87', '0.86', '0.85', '0.87', '0.85']
----------------------- Finished Computing Rashomon set ------------------------


In [7]:
smooth_trainer = SmoothTrainer(
    model,
    400,
    0.9,
    paradigm="DIL",
    seed=SEED,
    domain_map_fn=domain_map_fn
)

smooth_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=64,
    epochs=5,
    lr=0.01,
)
smooth_trainer.test(test_tasks)

smooth_trainer.compute_lid(test_tasks[0], bounded_model=interval_trainer.bounds[0])

smooth_trainer.train(
    train_tasks[1],
    val_tasks[1],
    batch_size=64,
    epochs=5,
    lr=0.01,
)
smooth_trainer.test(test_tasks)


Training Epochs: 100%|█| 5/5 [00:06<00:00,  1.30s/it, train_loss=0.0000, val_los


Test Results: [(0.0122, 0.9945), (0.923, 0.7743), (2.5359, 0.4075), (3.0403, 0.5447), (6.829, 0.199)] (Avg: (2.6681, 0.5840))


100%|████████████████████████████████████████| 100/100 [00:00<00:00, 654.68it/s]


Estimated success:  1.0  Lower bound:  0.9637833073548235
Certified L2 radius: 1.7964


100%|████████████████████████████████████████| 100/100 [00:00<00:00, 660.10it/s]


...

100%|████████████████████████████████████████| 100/100 [00:00<00:00, 659.52it/s]


Estimated success:  0.77  Lower bound:  0.6751412672248971
Certified L2 radius: 15.8954
Max rad:  28.634011572084738  inflate:  25
Last rad:  15.895415453527136  last inflate:  35


Training Epochs: 100%|█| 5/5 [00:09<00:00,  1.85s/it, train_loss=0.0044, val_los


Test Results: [(1.1046, 0.7473), (0.0141, 0.9965), (1.8765, 0.3743), (0.1813, 0.9431), (8.7015, 0.4354)] (Avg: (2.3756, 0.6993))


[(1.1046, 0.7473),
 (0.0141, 0.9965),
 (1.8765, 0.3743),
 (0.1813, 0.9431),
 (8.7015, 0.4354)]

### Class Incremental Training

In [8]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [10]:
interval_trainer = IntervalTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
)

interval_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=CONFIG["batch_size"],
    epochs=CONFIG["epochs"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)
interval_trainer.test(test_tasks)
interval_trainer.compute_rashomon_set(test_tasks[0])

Training Epochs:   0%| | 0/5 [00:00<?, ?it/s, val_loss=0.0000, val_acc=None, pro

Training Epochs: 100%|█| 5/5 [00:06<00:00,  1.27s/it, val_loss=0.0141, val_acc=0


Test Results: [(0.0358, 0.9865), (46.3004, 0.0), (39.2716, 0.0), (47.2765, 0.0), (43.7228, 0.0)] (Avg: (35.3214, 0.1973))
Initial acc constraint violation: -0.1251 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.86
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|█| 200/200 [00:09<00:00, 20.59it/s, size=1103.50, obj=0.178, min_soft_acc=1


Final bbox:  Obj=0.18,  Size=1103.50,  Min acc hard=0.88,  Min acc soft=0.88
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['22.49', '128.00', '330.09', '440.19', '553.35', '671.99', '781.32', '889.08', '996.72', '1103.50']
Checkpoint certificates: ['0.98', '0.98', '0.91', '0.95', '0.90', '0.88', '0.88', '0.88', '0.88', '0.88']
----------------------- Finished Computing Rashomon set ------------------------


In [15]:
smooth_trainer = SmoothTrainer(
    model,
    400,
    0.9,
    paradigm="CIL",
    seed=SEED,
)

smooth_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=64,
    epochs=5,
    lr=0.01,
)
smooth_trainer.test(test_tasks)

smooth_trainer.compute_lid(test_tasks[0], bounded_model=interval_trainer.bounds[0])

smooth_trainer.train(
    train_tasks[1],
    val_tasks[1],
    batch_size=64,
    epochs=5,
    lr=0.01,
)
smooth_trainer.test(test_tasks)


Training Epochs: 100%|█| 5/5 [00:06<00:00,  1.29s/it, train_loss=0.0001, val_los


Test Results: [(0.0106, 0.9965), (42.3224, 0.0), (43.194, 0.0), (46.1687, 0.0), (44.7789, 0.0)] (Avg: (35.2949, 0.1993))


100%|████████████████████████████████████████| 100/100 [00:00<00:00, 657.76it/s]


Estimated success:  1.0  Lower bound:  0.9637833073548235
Certified L2 radius: 1.7964


100%|████████████████████████████████████████| 100/100 [00:00<00:00, 634.95it/s]


...

100%|████████████████████████████████████████| 100/100 [00:00<00:00, 648.37it/s]


Estimated success:  1.0  Lower bound:  0.9637833073548235
Certified L2 radius: 116.7650
Max rad:  116.7649933014542  inflate:  65
Last rad:  116.7649933014542  last inflate:  65


Training Epochs: 100%|█| 5/5 [00:09<00:00,  1.90s/it, train_loss=0.0188, val_los


Test Results: [(8.4132, 0.008), (0.0236, 0.994), (20.2079, 0.0), (27.4394, 0.0), (29.3337, 0.0)] (Avg: (17.0836, 0.2004))


[(8.4132, 0.008),
 (0.0236, 0.994),
 (20.2079, 0.0),
 (27.4394, 0.0),
 (29.3337, 0.0)]